# 169. ColBERT：Late Interaction、MaxSim、候选检索与压缩怎样实现？

> **面试问题：为什么 ColBERT 不把文档压成一个向量？token-level MaxSim 怎样训练、建索引、召回和评估？**

## 先给结论

ColBERT 独立编码 query/document 的多向量表示，离线保存文档 token 向量；在线以每个 query token 对文档 token 的最大相似度求和。它保留细粒度匹配又避免 cross-encoder 在线联合编码，但索引体积、候选生成、padding mask、压缩与尾延迟是主要工程代价。

## 推荐回答主线

1. 先写出 L2 归一化 token embedding 与 `sum_i max_j q_i·d_j`，明确 padding/special token mask。
2. 用 pooled-vector 反例说明 late interaction 保存了哪些词粒度证据，再实现 in-batch 对比损失。
3. 候选阶段按 query token 查近邻并做 doc union，随后对候选执行精确 MaxSim；召回和重排分开评估。
4. 讨论 residual compression、索引字节、ACL/版本、Recall/MRR/nDCG、延迟分位和更新策略。

## 教学边界

Tiny encoder 只用 Embedding+Linear，候选索引用暴力 token 相似度，压缩用小型 centroid+int8 residual；它们展示合同，不替代 BERT、PLAID、GPU kernel 或分布式 ANN。

## 一手资料

- [ColBERT](https://arxiv.org/abs/2004.12832)
- [ColBERTv2](https://arxiv.org/abs/2112.01488)
- [PLAID](https://arxiv.org/abs/2205.09707)


In [ ]:
import hashlib
import json
import math
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

# 0 是 padding；小语料同时包含词粒度匹配和无关文档。
torch.manual_seed(169)
VOCAB, DIM = 32, 8
queries = torch.tensor([[1, 2, 0], [4, 5, 6]])
documents = torch.tensor([[1, 9, 2, 0], [4, 7, 5, 0], [8, 10, 11, 12], [6, 5, 4, 0]])
q_mask, d_mask = queries.ne(0), documents.ne(0)

assert queries.ndim == documents.ndim == 2
assert q_mask.sum().item() == 5
assert d_mask.shape == documents.shape


## 1. 独立编码多向量：padding 必须归零且不参与归一化

query 与 document 可以有不同 marker/最大长度，但最终每个有效 token 都投影到共同维度并 L2 归一化。归一化让点积成为 cosine；padding 行显式置零，后续仍需 mask，不能仅依赖零向量。


In [ ]:
class TinyColBERT(nn.Module):
    def __init__(self, vocab, dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab, dim, padding_idx=0)
        self.projection = nn.Linear(dim, dim, bias=False)

    def encode(self, token_ids, mask):
        projected = self.projection(self.embedding(token_ids))
        normalized = F.normalize(projected, p=2, dim=-1)
        return normalized * mask[..., None]

# 有效 token 范数为 1，padding 表示严格为零。
encoder = TinyColBERT(VOCAB, DIM)
q_vec = encoder.encode(queries, q_mask)
d_vec = encoder.encode(documents, d_mask)
assert q_vec.shape == (2, 3, DIM)
assert torch.allclose(q_vec[q_mask].norm(dim=-1), torch.ones(q_mask.sum()), atol=1e-6)
assert d_vec[~d_mask].abs().sum().item() == 0.0


## 2. MaxSim：每个 query token 找最匹配文档 token，再求和

批量得分张量可写成 `[Q,D,Lq,Ld]`。先把无效 document token 设为 `-inf`，沿 Ld 取 max；再用 query mask 把 padding query 清零并求和。空文档需要单独拒绝，避免整行 `-inf`。


In [ ]:
def maxsim_scores(query_vectors, document_vectors, query_mask, document_mask):
    similarity = torch.einsum("qie,dje->qdij", query_vectors, document_vectors)
    similarity = similarity.masked_fill(~document_mask[None, :, None, :], -torch.inf)
    best_per_query_token = similarity.max(dim=-1).values
    best_per_query_token = torch.where(query_mask[:, None, :], best_per_query_token, torch.zeros_like(best_per_query_token))
    return best_per_query_token.sum(dim=-1), best_per_query_token

# 得分矩阵覆盖每个 query-document 对，padding query 不贡献分数。
scores, token_scores = maxsim_scores(q_vec, d_vec, q_mask, d_mask)
assert scores.shape == (2, 4)
assert token_scores.shape == (2, 4, 3)
assert token_scores[0, :, 2].abs().sum().item() == 0.0


## 3. 为什么不只做 mean pooling：相反 token 可在均值中抵消

单向量双塔把所有 token 压到一个向量，细粒度匹配可能被平均抵消。下面构造两个文档均值相同、但 token 级覆盖不同的反例；MaxSim 能识别每个 query token 是否都有证据。


In [ ]:
# 两维手工向量：query 需要 x 与 y 两个方向。
manual_q = torch.tensor([[[1.0, 0.0], [0.0, 1.0]]])
manual_d = torch.tensor([
    [[1.0, 0.0], [0.0, 1.0]],
    [[0.5, 0.5], [0.5, 0.5]],
])
manual_mask_q = torch.ones(1, 2, dtype=torch.bool)
manual_mask_d = torch.ones(2, 2, dtype=torch.bool)
late, _ = maxsim_scores(manual_q, manual_d, manual_mask_q, manual_mask_d)
pooled_q = manual_q.mean(1)
pooled_d = manual_d.mean(1)
pooled = pooled_q @ pooled_d.T

# 两个文档 pooled 得分相同，但精确覆盖两个方向的文档 MaxSim 更高。
assert torch.allclose(pooled[0, 0], pooled[0, 1])
assert late[0, 0] > late[0, 1]
assert late.tolist() == [[2.0, 1.0]]


## 4. In-batch negatives：对角正例只是数据合同，不是自然事实

若 batch 第 i 个 query 与第 i 个 document 是正例，可对 Q×D MaxSim 矩阵做交叉熵。真实训练还要处理多个正例、false negative、跨设备 negatives、温度和去重；否则相同答案文档会被误罚。


In [ ]:
def in_batch_loss(score_matrix, positive_doc_ids, temperature=1.0):
    return F.cross_entropy(score_matrix / temperature, positive_doc_ids)

# 选前两个文档与两个 query 配对，loss 有限且能回传到 encoder。
train_scores, _ = maxsim_scores(q_vec, d_vec[:2], q_mask, d_mask[:2])
positive_ids = torch.arange(2)
loss = in_batch_loss(train_scores, positive_ids, temperature=0.7)
encoder.zero_grad(set_to_none=True)
loss.backward()
assert torch.isfinite(loss)
assert encoder.embedding.weight.grad is not None
assert torch.isfinite(encoder.projection.weight.grad).all()


## 5. 候选检索：逐 query token 取近邻文档并做 union

全库精确 MaxSim 太贵。可让每个 query token 从 token ANN 取 top-k，合并 doc id 形成候选，再只对候选算完整 MaxSim。`k` 越小越快但可能漏掉总体相关文档，因此必须画候选 recall—访问量曲线。


In [ ]:
@torch.no_grad()
def token_candidate_docs(single_query, single_qmask, all_documents, all_dmask, per_token_k=2):
    candidates = set()
    for qtoken in single_query[single_qmask]:
        token_sim = torch.einsum("e,dje->dj", qtoken, all_documents)
        token_sim = token_sim.masked_fill(~all_dmask, -torch.inf)
        doc_best = token_sim.max(-1).values
        candidates.update(torch.topk(doc_best, min(per_token_k, len(doc_best))).indices.tolist())
    return sorted(candidates)

# 候选是合法 doc id，预算不超过 query_token_count*k，且至少找回精确 top-1。
candidate_ids = token_candidate_docs(q_vec[0], q_mask[0], d_vec, d_mask, per_token_k=2)
exact_top1 = int(scores[0].argmax())
assert exact_top1 in candidate_ids
assert all(0 <= doc_id < len(documents) for doc_id in candidate_ids)
assert len(candidate_ids) <= int(q_mask[0].sum()) * 2


## 6. Residual compression：centroid id + 量化残差

ColBERTv2 通过 residual compression 降低多向量索引体积。教学版给定 centroid，保存最近 centroid id、每 token scale 和 int8 residual；解码后重新归一化。实际码本训练、bit packing 与 PLAID 剪枝更复杂。


In [ ]:
def compress_residual(vectors, centroids):
    distance = ((vectors[:, None, :] - centroids[None, :, :]) ** 2).sum(-1)
    ids = distance.argmin(1)
    residual = vectors - centroids[ids]
    scale = residual.abs().amax(1).clamp_min(1e-8) / 127
    quantized = torch.round(residual / scale[:, None]).clamp(-127, 127).to(torch.int8)
    return ids, scale, quantized

def decompress_residual(ids, scale, quantized, centroids):
    reconstructed = centroids[ids] + quantized.float() * scale[:, None]
    return F.normalize(reconstructed, dim=-1)

# 重建形状保持不变，误差有限，教学压缩字节少于 float32 原向量。
flat_valid = d_vec.detach()[d_mask]
centroids = flat_valid[:3].clone()
centroid_ids, scales, residual_q = compress_residual(flat_valid, centroids)
reconstructed = decompress_residual(centroid_ids, scales, residual_q, centroids)
original_bytes = flat_valid.numel() * 4
compressed_bytes = centroid_ids.numel() * 2 + scales.numel() * 4 + residual_q.numel()
assert reconstructed.shape == flat_valid.shape
assert (reconstructed - flat_valid).norm(dim=-1).mean() < 0.03
assert compressed_bytes < original_bytes


## 7. 检索评测：候选 recall 与最终 MRR 必须分层归因

候选漏掉 gold 时，重排器再强也救不回；候选包含 gold 但排名差才是 MaxSim/encoder 问题。空 gold 查询应从指标中显式排除或单独报告，不能用 rank=∞ 悄悄拉低均值。


In [ ]:
def retrieval_metrics(rankings, gold_sets, k):
    recalls, reciprocal = [], []
    for ranking, gold in zip(rankings, gold_sets):
        if not gold:
            continue
        recalls.append(len(set(ranking[:k]) & gold) / len(gold))
        ranks = [i + 1 for i, doc in enumerate(ranking) if doc in gold]
        reciprocal.append(1 / min(ranks) if ranks else 0.0)
    return float(np.mean(recalls)), float(np.mean(reciprocal))

# 两个 query 的 gold 可复现；指标范围合法，交换首位会改变 MRR。
rankings = torch.argsort(scores.detach(), dim=1, descending=True).tolist()
gold = [{0}, {1, 3}]
recall2, mrr = retrieval_metrics(rankings, gold, k=2)
assert 0 <= recall2 <= 1
assert 0 <= mrr <= 1
assert retrieval_metrics([[0, 1], [1, 3]], gold, 2)[1] == 1.0


## 8. 索引制品：encoder、tokenizer、码本、ACL 快照必须原子绑定

在线 query encoder 必须与离线文档向量同版本；压缩码本变化需要重建或双读迁移。检索前后都执行 ACL，日志记录候选数、token probes、精确 MaxSim 数、压缩版本与超时降级。


In [ ]:
@dataclass(frozen=True)
class ColBERTIndex:
    encoder_hash: str
    tokenizer_hash: str
    centroid_hash: str
    acl_snapshot: str
    dimension: int

def index_digest(manifest):
    return hashlib.sha256(json.dumps(asdict(manifest), sort_keys=True).encode()).hexdigest()

# 任一核心版本变化都产生新摘要，维度必须与在线 query 向量一致。
manifest = ColBERTIndex("enc-v4", "tok-v2", "cent-v7", "acl-2026-07", DIM)
digest = index_digest(manifest)
assert len(digest) == 64
assert manifest.dimension == q_vec.shape[-1]
assert digest != index_digest(ColBERTIndex("enc-v5", "tok-v2", "cent-v7", "acl-2026-07", DIM))


## 面试收束与生产替换点

完整回答不要停在算法名：先说清业务目标、输入输出与信任边界，再给核心数据结构/公式和可执行 oracle，最后落到离线切片、线上 SLO、成本、安全、版本、灰度与回滚。这里的受控实现用于解释机制和发现反例；真实模型编码器、分布式索引、协议 SDK、安全沙箱、监控与持久层应作为可替换组件，并用同一合同验收。

典型追问包括：数据规模扩大后瓶颈在哪？近似步骤损失了什么？哪个状态必须持久化？超时或部分失败怎样降级？版本错配为何不能静默兼容？离线指标上升是否来自污染、权限泄漏、评测器偏差或重复样本？
